# Phase 2 verification - `myroom`

Confirms the strip-to-core **Phase 2** changes did not regress static quality:
- `fourier_K=0` in static mode (freed GPU memory; params had LR=0 anyway)
- static export now writes **one** clean frame instead of running the placeholder deform per timestamp

Closes task **P2-V**. Runtime: **A100 GPU** (Runtime -> Change runtime type). ~10-25 min.


## 0. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Clone repo (branch `chore/strip-to-core`)

In [ ]:
import os
REPO_URL = "https://github.com/mehmettahacumurcu/gaussian-splatter.git"
BRANCH   = "chore/strip-to-core"   # Phase 2 work lives here, NOT main
if not os.path.exists("gaussian-splatter"):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL}
%cd gaussian-splatter
!git log --oneline -1

## 2. Environment (deps + gsplat JIT build)

In [ ]:
!bash colab/bootstrap.sh

In [ ]:
# If this errors about numpy: Runtime -> Restart session, then re-run THIS cell only.
import torch, gsplat
print("torch", torch.__version__, "| gsplat", gsplat.__version__,
      "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 3. Data - mount Drive and link `myroom`

Upload the pre-processed `myroom` scene to your Drive once (folders `frames/` + `colmap/`;
add `depth/` only if you plan to pass `--foundation`). Edit `DRIVE_MYROOM` below to match.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MYROOM = "/content/drive/MyDrive/4dgs/myroom"   # <-- EDIT to your Drive path
import os
os.makedirs("data", exist_ok=True)
target = "data/myroom"
if not (os.path.islink(target) or os.path.exists(target)):
    os.symlink(DRIVE_MYROOM, target)
print("linked:", os.path.realpath(target))
!ls -la data/myroom

## 4. Run static 3DGS + NVS eval

`balanced` preset is plenty for a no-regression check. Add `--foundation` only if you
uploaded `depth/` and want depth supervision (closer to the original 29 dB run).


In [ ]:
!python scripts/static_3dgs.py --scene myroom --preset balanced --nvs-eval

## 5. Verify (no-regression gate)

In [ ]:
import sys
sys.path.insert(0, ".")
from colab.verify_helpers import check_phase2, gpu_mem_summary
gpu_mem_summary()
ok = check_phase2("myroom", baseline_psnr=29.0, min_psnr=27.0)
print("\nP2-V:", "PASS - Phase 2 confirmed" if ok else "FAIL - investigate above")

## 6. Save results to Drive

In [ ]:
from colab.verify_helpers import copy_results_to_drive
copy_results_to_drive("myroom", "/content/drive/MyDrive/4dgs/results")